In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
project_path = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)
from utils.transformations import reusable

In [0]:
df_user = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "parquet")
         .option(
             "cloudFiles.schemaLocation",
             "abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimUser/schema"
         )
         .option("schemaEvolutionMode","addNewColumns")\
         .load(
             "abfss://bronze@datalakestoragevishnu.dfs.core.windows.net/DimUser"
         )
)

In [0]:
df_user = df_user.writeStream.format('delta').outputMode('append').option('checkpointLocation','abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimUser/checkpoint')\
.option("mergeSchema","true")\
.trigger(once=True).option('path','abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimUser/data').toTable('spotify_cata.default.DimUser')

In [0]:
df_user = df_user.withColumn('user_name',upper(col('user_name')))

In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user = df_user_obj.dropColumns(df_user,['end_date'])
df_user = df_user.dropDuplicates(['user_id'])

In [0]:
print(df_user.isStreaming)

In [0]:
display(df_user)

In [0]:
df_artist = spark.readStream.format("cloudFiles")\
                            .option("cloudFiles.format","parquet")\
                            .option("cloudFiles.schemaLocation","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimArtist/schema")\
                            .option("schemaEvolutionMode","addNewColumns")\
                            .load("abfss://bronze@datalakestoragevishnu.dfs.core.windows.net/DimArtist")

In [0]:
df_art_obj = reusable()
df_artist = df_art_obj.dropColumns(df_artist,['_rescued_data'])
df_artist = df_artist.dropDuplicates(['artist_id'])

In [0]:
print(df_artist.isStreaming)

In [0]:
df_artist.writeStream.format('delta')\
                    .outputMode('append')\
                    .option('checkpointLocation','abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimArtist/checkpoint')\
                         .option("mergeSchema","true")\
                    .trigger(once=True)\
                    .option("path","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimArtist/data")\
                    .toTable('spotify_cata.default.DimArtist')

In [0]:
%sql
select current_catalog()

##DimTrack

In [0]:
df_track_stream = spark.readStream.format("cloudFiles")\
                            .option("cloudFiles.format","parquet")\
                            .option("cloudFiles.schemaLocation","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimTrack/schema")\
                            .option("schemaEvolutionMode","addNewColumns")\
                            .load("abfss://bronze@datalakestoragevishnu.dfs.core.windows.net/DimTrack")

In [0]:
print(df_track.isStreaming)

In [0]:
df_track_stream = df_track_stream.withColumn("durationFlag",when(col("duration_sec")<150, "low")\
                                    .when(col("duration_sec")<300,"medium")\
                                        .otherwise("high"))

In [0]:
df_track_stream = df_track_stream.withColumn("track_name",regexp_replace(col("track_name"),"-"," "))

In [0]:
df_track_stream = reusable().dropColumns(df_track_stream,['_rescued_data'])

In [0]:
df_track_stream.writeStream.format('delta')\
                    .outputMode('append')\
                    .option('checkpointLocation','abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimTrack/checkpoint')\
                         .option("mergeSchema","true")\
                    .trigger(once=True)\
                    .option("path","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimTrack/data")\
                    .toTable('spotify_cata.default.DimTrack')

#DimDate

In [0]:
df_date = spark.readStream.format("cloudFiles")\
                            .option("cloudFiles.format","parquet")\
                            .option("cloudFiles.schemaLocation","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimDate/schema")\
                            .option("schemaEvolutionMode","addNewColumns")\
                            .load("abfss://bronze@datalakestoragevishnu.dfs.core.windows.net/DimDate")

In [0]:
df_date = reusable().dropColumns(df_date, ['_rescued_data'])

In [0]:
df_date.writeStream.format("delta")\
                   .option("checkpointLocation","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimDate/checkpoint")\
                   .option("mergeSchema","true")\
                   .trigger(once=True)\
                   .option("path","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/DimDate/data")\
                   .toTable('spotify_cata.default.DimDate')

#Fact stream

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
                            .option("cloudFiles.format","parquet")\
                            .option("cloudFiles.schemaLocation","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/FactStream/schema")\
                            .option("schemaEvolutionMode","addNewColumns")\
                            .load("abfss://bronze@datalakestoragevishnu.dfs.core.windows.net/FactStream")

In [0]:
df_fact = reusable().dropColumns(df_fact, ['_rescued_data'])

In [0]:
df_fact.writeStream.format("delta")\
                   .option("checkpointLocation","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/FactStream/checkpoint")\
                   .option("mergeSchema","true")\
                   .trigger(once=True)\
                   .option("path","abfss://silver@datalakestoragevishnu.dfs.core.windows.net/FactStream/data")\
                   .toTable('spotify_cata.default.FactStream')
